# Offer 데이터 기반 제구 성공 확률 모델 전략

**TL;DR.** 핵심은 `asof_*` 누적값에서 과거 시즌 종료 시점의 정적 누적값을 빼서 **현재 시즌의 투수·타자 상태를 투구 직전 기준으로 복원**하는 것입니다. 여기에 CatBoost(고카디널리티 ID·상호작용), LightGBM(수치·Trackman 집계), 시간 순 교차검증, 2025 고정 사전확률 보정을 결합합니다. 2024 홀드아웃에서 RF 베이스라인 `0.690675` 대비 최종 후보는 로그손실 `0.688286`을 기록했습니다.

모든 피처는 현재 투구 결과나 다른 테스트 행을 사용하지 않습니다. 이 노트북은 원천 데이터 점검, 핵심 피처의 시간 안전성 검증, 실험 결과 기록을 재현합니다.

## 1. 범위와 검증 원칙

- 학습: 2019~2024 `train.csv`, 보조 이력: 2019~2024 `trackman_history.csv`
- 주 검증: 2019~2023 학습 → 2024 검증
- 강건성 검증: 2022, 2023, 2024를 각각 미래 홀드아웃으로 사용
- 평가 기준: binary logloss(주), ROC-AUC(순위 품질 보조)
- 테스트 행끼리 집계·정렬·롤링하지 않고, 학습에서 만든 정적 메타데이터와 각 테스트 행의 공식 `asof_*` 값만 사용

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / 'offer' / 'data' / 'train.csv').exists():
    raise FileNotFoundError('저장소 루트에서 노트북을 실행하세요.')
TRAIN_PATH = ROOT / 'offer' / 'data' / 'train.csv'
TEST_PATH = ROOT / 'offer' / 'data' / 'test.csv'
TRACKMAN_PATH = ROOT / 'offer' / 'data' / 'trackman_history.csv'

train_cols = [
    'row_id', 'season', 'game_type', 'pitcher_id', 'batter_id',
    'asof_pitcher_n', 'asof_pitcher_success_rate',
    'asof_batter_n', 'asof_batter_success_rate', 'control_success'
]
train = pd.read_csv(TRAIN_PATH, usecols=train_cols)
test_header = pd.read_csv(TEST_PATH, nrows=0)
trackman_cols = ['season', 'pitcher_trackman_id', 'batter_trackman_id', 'pitch_type_group',
                 'rel_speed', 'spin_rate', 'induced_vert_break', 'horz_break',
                 'extension', 'rel_height', 'rel_side', 'zone_speed']
trackman = pd.read_csv(TRACKMAN_PATH, usecols=trackman_cols)
print('train:', train.shape, 'test sample columns:', len(test_header.columns), 'trackman:', trackman.shape)

train: (1475092, 10) test sample columns: 48 trackman: (1793078, 12)


## 2. 데이터 품질과 분포

행 식별자·타깃·연도 범위를 검증하고, 운영 환경에서 기대하는 스키마와 일치하는지 확인합니다.

In [2]:
assert len(train) == 1_475_092
assert len(test_header.columns) == 48
assert train['row_id'].is_unique
assert train['control_success'].isin([0, 1]).all()
assert sorted(train['season'].unique().tolist()) == [2019, 2020, 2021, 2022, 2023, 2024]
assert sorted(trackman['season'].unique().tolist()) == [2019, 2020, 2021, 2022, 2023, 2024]
inventory = pd.DataFrame([
    {'dataset': 'train', 'rows': len(train), 'columns': 49, 'years': '2019-2024'},
    {'dataset': 'test(sample)', 'rows': len(pd.read_csv(TEST_PATH)), 'columns': len(test_header.columns), 'years': '2025'},
    {'dataset': 'trackman_history', 'rows': len(trackman), 'columns': 30, 'years': '2019-2024'},
])
inventory

,dataset,rows,columns,years
0,train,1475092,49,2019-2024
1,test(sample),5,48,2025
2,trackman_history,1793078,30,2019-2024


In [3]:
season_rate = (train.groupby(['season', 'game_type'], observed=True)['control_success']
               .agg(['mean', 'size']).reset_index())
overall = (train.groupby('season')['control_success'].agg(['mean', 'size']).reset_index()
           .assign(game_type='전체'))
drift = pd.concat([overall, season_rate], ignore_index=True)
drift['success_rate'] = drift.pop('mean').round(6)
drift.sort_values(['season', 'game_type'])

,season,size,game_type,success_rate
6,2019,25786,F,0.689250
7,2019,211627,R,0.549490
0,2019,237413,전체,0.564670
8,2020,23213,F,0.587774
9,2020,220874,R,0.526925
1,2020,244087,전체,0.532712
10,2021,25861,F,0.703840
11,2021,221227,R,0.512763
2,2021,247088,전체,0.532762
12,2022,30448,F,0.708749


## 3. 핵심 피처: 현재 시즌 누적의 시간 안전 복원

`asof_pitcher_success_rate × asof_pitcher_n`은 현재 투구 직전까지의 커리어 성공 횟수를 복원합니다. 각 투수·시즌 첫 행의 누적값(이전 시즌까지의 정적 값)을 빼면 현재 시즌의 직전 투구까지 성공 횟수와 투구 수가 나옵니다. 아래 검증은 이 계산이 현재 행의 타깃을 포함하지 않음을 전 행에서 확인합니다.

In [4]:
career_success_before = np.rint(np.nan_to_num(
    train['asof_pitcher_success_rate'].to_numpy() * train['asof_pitcher_n'].to_numpy(),
    nan=0.0,
)).astype('int64')
g = train.groupby(['pitcher_id', 'season'], sort=False, observed=True)
season_start_n = g['asof_pitcher_n'].transform('first').to_numpy()
season_start_success = pd.Series(career_success_before).groupby(
    [train['pitcher_id'], train['season']], sort=False
).transform('first').to_numpy()
derived_n = train['asof_pitcher_n'].to_numpy() - season_start_n
derived_success = career_success_before - season_start_success
true_n = g.cumcount().to_numpy()
true_success = (g['control_success'].cumsum() - train['control_success']).to_numpy()
proof = {
    'pitch_count_mismatches': int(np.count_nonzero(derived_n != true_n)),
    'success_count_mismatches': int(np.count_nonzero(derived_success != true_success)),
    'rows_checked': len(train),
}
assert proof['pitch_count_mismatches'] == 0
assert proof['success_count_mismatches'] == 0
proof

{'pitch_count_mismatches': 0,
 'success_count_mismatches': 0,
 'rows_checked': 1475092}

## 4. Trackman 활용 설계

익명 ID가 직접 연결되지 않으므로, 시즌별 투구 수·구종 구성·손잡이·활동 시즌 패턴을 이용해 메인 투수와 Trackman 투수를 매칭합니다. 상호 최근접, 거리 임계값, 차순위와의 마진을 만족한 매칭만 채택하고 신뢰도를 피처로 함께 넣습니다. 물리량은 반드시 예측 연도 이전 시즌까지만 누적해 구종군별 평균·표준편차·표본 수로 저장합니다. 2024 홀드아웃에서 완화된 신뢰도 매칭은 약 60.9%의 행에 후보를 제공했고, 결합 LightGBM의 로그손실을 추가로 개선했습니다.

In [5]:
physical = ['rel_speed', 'spin_rate', 'induced_vert_break', 'horz_break',
            'extension', 'rel_height', 'rel_side', 'zone_speed']
trackman_summary = {
    'rows': len(trackman),
    'pitchers': int(trackman['pitcher_trackman_id'].nunique()),
    'batters': int(trackman['batter_trackman_id'].nunique()),
    'pitch_groups': int(trackman['pitch_type_group'].nunique(dropna=True)),
    'max_physical_missing_rate': float(trackman[physical].isna().mean().max()),
}
trackman_summary

{'rows': 1793078,
 'pitchers': 906,
 'batters': 913,
 'pitch_groups': 4,
 'max_physical_missing_rate': 0.00695173327652227}

## 5. 2024 홀드아웃 실험 결과

아래 값은 동일한 2019~2023 학습/2024 검증 분할에서 실행한 기록입니다. 로그손실이 낮을수록 좋습니다. 최종 후보는 CatBoost와 LightGBM의 확률을 혼합한 뒤, 학습 데이터에서만 추정한 2025 리그별 사전확률로 로짓 절편을 고정 보정하는 구조입니다.

In [6]:
benchmarks = pd.DataFrame([
    ('RandomForest baseline', 0.6906754394, 0.5394623),
    ('기존 weighted LightGBM', 0.6892090375, 0.5485300),
    ('Trackman LightGBM blend', 0.6890501070, 0.5497330),
    ('현재 시즌 피처 LightGBM', 0.6887766790, 0.5514990),
    ('현재 시즌 + Trackman LightGBM', 0.6886673000, 0.5524850),
    ('현재 시즌 + ID CatBoost', 0.6884680700, 0.5526700),
    ('CatBoost/LightGBM raw blend', 0.6884394310, 0.5532790),
    ('리그별 보정 blend', 0.6882857860, 0.5531100),
], columns=['model', 'logloss', 'auc'])
reference = benchmarks.loc[benchmarks['model'].eq('기존 weighted LightGBM'), 'logloss'].iloc[0]
benchmarks['improvement_vs_weighted_lgb_x1e4'] = ((reference - benchmarks['logloss']) * 10_000).round(3)
benchmarks.sort_values('logloss')

,model,logloss,auc,improvement_vs_weighted_lgb_x1e4
7,리그별 보정 blend,0.688286,0.553110,9.233
6,CatBoost/LightGBM raw blend,0.688439,0.553279,7.696
5,현재 시즌 + ID CatBoost,0.688468,0.552670,7.410
4,현재 시즌 + Trackman LightGBM,0.688667,0.552485,5.417
3,현재 시즌 피처 LightGBM,0.688777,0.551499,4.324
2,Trackman LightGBM blend,0.689050,0.549733,1.589
1,기존 weighted LightGBM,0.689209,0.548530,0.000
0,RandomForest baseline,0.690675,0.539462,-14.664


In [7]:
rolling = pd.DataFrame([
    (2022, 0.6796081, 0.6789633, 0.577778, 0.581674),
    (2023, 0.6948794, 0.6936811, 0.526252, 0.533216),
    (2024, 0.6892217, 0.6887767, 0.548353, 0.551499),
], columns=['holdout_year', 'base_logloss', 'seasonal_logloss', 'base_auc', 'seasonal_auc'])
rolling['logloss_gain_x1e4'] = ((rolling['base_logloss'] - rolling['seasonal_logloss']) * 10_000).round(3)
rolling

,holdout_year,base_logloss,seasonal_logloss,base_auc,seasonal_auc,logloss_gain_x1e4
0,2022,0.679608,0.678963,0.577778,0.581674,6.448
1,2023,0.694879,0.693681,0.526252,0.533216,11.983
2,2024,0.689222,0.688777,0.548353,0.551499,4.450


## 6. 권장 최종 구조와 실행 순서

1. **시간 안전 피처 저장**: 연도별 이전 시즌 종료 누적값, 현재 시즌 누적/베이지안 축소율, 이전 시즌만으로 만든 투수×상황·타자×상황 타깃 인코딩을 생성합니다.
2. **Trackman 정적 메타데이터**: 2024년까지의 ID 매칭, 신뢰도, 구종군별 물리 프로파일을 작은 테이블로 굳혀 모델과 함께 패키징합니다. 추론 때 337MB 원본을 읽지 않습니다.
3. **이질적 앙상블**: CatBoost는 선수 ID와 범주 상호작용, LightGBM은 연속형 누적·Trackman 피처를 담당합니다. 2024에서 CatBoost 60~80% 범위의 혼합이 우수했습니다.
4. **고정 확률 보정**: 2019~2024의 리그별 추세에서 산출한 2025 사전확률 `R≈0.4848`, `F≈0.4552`를 로짓 절편으로 적용합니다. 테스트 타깃이나 테스트 행 집계는 사용하지 않습니다.
5. **최종 선택**: 2022~2024 롤링 홀드아웃 평균·최악 성능을 함께 보고 seed/트리 수/혼합비를 고정한 뒤 2019~2024 전체로 재학습합니다.
6. **제출 QA**: 245,789행 순서·ID 완전 일치, 확률 유한값 및 `[0,1]`, 10분 제한, 28GB RAM을 실제 제출 컨테이너와 유사한 환경에서 검증합니다. 행이 누락되면 즉시 오류를 내도록 합니다.

숨은 2025 분포가 공개되지 않았으므로 이 결과는 리더보드 성능의 보장이 아니라, 공개 데이터에서 확인된 가장 강한 시간 순 오프라인 근거입니다.